# A Dead by Daylight adatainak elemzése


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px


DATA = Path("dataset")
DATA_URL = "https://raw.githubusercontent.com/nemethbalu06-svg/Python-projekt/main/dataset"
RARITIES = {
    "Common": "Gyakori",
    "Uncommon": "Nem gyakori",
    "Rare": "Ritka",
    "Very Rare": "Nagyon ritka",
    "Ultra Rare": "Rendkívül ritka",
}


def read(name):
    filename = f"{name}.csv"
    source = DATA / filename if DATA.is_dir() else f"{DATA_URL}/{filename}"
    return pd.read_csv(source)


def counts(data, column, label):
    return data[column].map(RARITIES).value_counts().rename_axis(label).reset_index(name="Darabszám")

## Karakterek

Hány gyilkos és túlélő szerepel az adatokban?

In [ ]:
killers = read("killers")
survivors = read("survivors")
characters = pd.DataFrame({
    "Szerepkör": ["Gyilkos", "Túlélő"],
    "Darabszám": [killers["killer"].nunique(), survivors["survivor"].nunique()],
})
print(characters.to_string(index=False))
px.bar(characters, x="Szerepkör", y="Darabszám", title="Karakterek száma szerepkörönként").show()

## Képességek

Az általános képességek külön szerepelnek a diagramon. A karakterenkénti átlagba csak a karakterhez tartozó képességek számítanak bele.

In [ ]:
perk_counts = []
for role, label, roster in [("killer", "Gyilkos", killers), ("survivor", "Túlélő", survivors)]:
    perks = read(f"{role}Perks")
    totals = perks.groupby(role)["perk"].nunique()
    character_totals = totals.reindex(roster[role], fill_value=0)
    perk_counts.append(totals.rename(index={"General": "Általános"}).rename_axis("Karakter").reset_index(name="Darabszám")
                       .assign(Szerepkör=label))
    average = np.mean(character_totals.to_numpy())
    print(f"Átlagos képességszám karakterenként ({label}): {average:.2f}")
perks_by_character = pd.concat(perk_counts, ignore_index=True)
px.bar(perks_by_character, x="Darabszám", y="Karakter", title="Karakterhez kötött és általános képességek",
       color="Szerepkör", barmode="group", height=1050).show()

## Pályák

Hány különböző pálya tartozik az egyes realmekhez?

In [ ]:
maps = read("maps")
realms = read("realms")
maps_by_realm = maps.groupby("realm")["map"].nunique().reindex(realms["realm"], fill_value=0)
maps_by_realm = maps_by_realm.sort_values(ascending=False).reset_index(name="Darabszám")
px.bar(maps_by_realm, x="realm", y="Darabszám", labels={"realm": "Realm"}, title="Pályák száma realmenként").show()

## Tárgyak és kiegészítők

A diagram ritkaság szerint mutatja a túlélők tárgyait, valamint a túlélők és a gyilkosok kiegészítőit.

In [ ]:
gear = pd.concat([
    counts(read("survivorItems"), "rarity", "Ritkaság").assign(Kategória="Túlélők tárgyai"),
    counts(read("survivorItemAddons"), "rarity", "Ritkaság").assign(Kategória="Túlélők kiegészítői"),
    counts(read("killerAddons"), "rarity", "Ritkaság").assign(Kategória="Gyilkosok kiegészítői"),
], ignore_index=True)
px.bar(gear, x="Kategória", y="Darabszám", title="Tárgyak és kiegészítők ritkaság szerint", color="Ritkaság").show()

## Felajánlások

A diagram a felajánlások számát szerepkör és ritkaság szerint mutatja. Egy felajánlás mindkét szerepkörnél szerepelhet.

In [ ]:
offerings = pd.concat([
    counts(read("killerOfferings"), "rarity", "Ritkaság").assign(Szerepkör="Gyilkos"),
    counts(read("survivorOfferings"), "rarity", "Ritkaság").assign(Szerepkör="Túlélő"),
], ignore_index=True)
px.bar(offerings, x="Szerepkör", y="Darabszám", title="Felajánlások szerepkör és ritkaság szerint", color="Ritkaság").show()